# Scrapy
- 웹사이트에서 데이터 수집을 위한 오픈소스 파이썬 프레임워크
- 멀티스레딩으로 데이터 수집
- daum news 상품데이터 수집

In [4]:
# install scrapy
!pip install scrapy

In [6]:
import scrapy, requests
from scrapy.http import TextResponse

## 1. make project

In [8]:
!scrapy startproject news

New Scrapy project 'news', using template directory 'C:\Users\User\anaconda3\Lib\site-packages\scrapy\templates\project', created in:
    C:\Users\User\aivle6-practice\web-crawling\news

You can start your first spider with:
    cd news
    scrapy genspider example example.com


In [21]:
!tree news /F

폴더 PATH의 목록입니다.
볼륨 일련 번호는 385B-BB36입니다.
C:\USERS\USER\AIVLE6-PRACTICE\WEB-CRAWLING\NEWS
│  scrapy.cfg
│  
└─news
    │  items.py
    │  middlewares.py
    │  pipelines.py
    │  settings.py
    │  __init__.py
    │  
    └─spiders
            __init__.py
            


- scrapy structure
    - items : 데이터의 모양 정의 //MVC의 model
    - middewares : 수집할때 header 정보와 같은 내용을 설정
    - pipelines : 데이터를 수집한 후에 코드를 실행
    - settings : robots.txt 규칙, 크롤링 시간 텀등을 설정
    - spiders : 크롤링 절차를 정의

## 2. xpath
- link, contents

In [5]:
import scrapy, requests
from scrapy.http import TextResponse # xpath를 쓰기 위함

In [30]:
url = 'https://news.daum.net/'
response = requests.get(url)
response = TextResponse(response.url, body=response.text, encoding='utf-8')
response

<200 https://news.daum.net/>

In [39]:
selector = '/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a'
links = response.xpath(selector)
len(links), links[0]

(20,
 <Selector query='/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a' data='<a href="https://v.daum.net/v/2024092...'>)

In [47]:
selector = '/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a/@href'
links = response.xpath(selector).extract()
len(links), links[0]

(20, 'https://v.daum.net/v/20240923151053270')

In [55]:
link = links[0]
response = requests.get(link)
response = TextResponse(response.url, body=response.text, encoding='utf-8')
response

<200 https://v.daum.net/v/20240923151053270>

In [63]:
title = response.xpath('//*[@id="mArticle"]/div[1]/h3/text()')[0].extract()
title

"전국 주택 2가구 중 1가구 이상은 '준공 20년 이상'"

## 3. items.py
- Data Model

In [67]:
# %load news/news/items.py

In [71]:
%%writefile news/news/items.py
import scrapy

class NewsItem(scrapy.Item):
    title = scrapy.Field()
    link = scrapy.Field()

Overwriting news/news/items.py


## 4. spider.py
- wirte crawling process

In [121]:
%%writefile news/news/spiders/spider.py
import scrapy
from news.items import NewsItem

class NewsSpider(scrapy.Spider):
    name = 'news'
    allow_domain = ['daum.net']
    start_urls = ['https://news.daum.net']

    def parse(self, response):
        selector = '/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a/@href'
        links = response.xpath(selector).extract()
        for link in links:
            yield scrapy.Request(link, callback=self.parse_content) # yield : 여러번 리턴할 때 사용

    def parse_content(self, response):
        item = NewsItem()
        item['link'] = response.url
        item['title'] = response.xpath('//*[@id="mArticle"]/div[1]/h3/text()')[0].extract()
        yield item

Overwriting news/news/spiders/spider.py


In [105]:
#yield 실습
def echo():
    yield 1
    yield 2
    yield 3

In [107]:
echo()

<generator object echo at 0x0000022F49244B40>

In [109]:
e = echo()

In [115]:
next(e)

3

## 5. run scrapy
- news 디렉토리에서 아래의 커멘드 실행
- scrapy crawl news -o news.csv

In [83]:
%pwd

'C:\\Users\\User\\aivle6-practice\\web-crawling'